# AR(p) processes and forward sliding-window embeddings

This notebook simulates a stationary AR(p) process

$$X_t = c + \phi_1X_{t-1}+\cdots+\phi_pX_{t-p}+\varepsilon_t,$$

using Gaussian, uniform, Laplace, Student-$t$, Cauchy, or centered Pareto innovations. It plots the time series and the forward embeddings

$$Y_t^{(k)}=(X_t,X_{t+1},\ldots,X_{t+k-1})$$

for $k=2$ and $k=3$. The 3D Plotly figure can be rotated, zoomed, and panned.

Geometric overlays:

- **Middle hyperplane:** the zero-innovation AR relation.
- **Uniform innovations:** the two border hyperplanes obtained by adding the lower/upper innovation bounds.
- **Gaussian innovations:** a highest-density-region (HDR) ellipse for $k=2$ and ellipsoid for $k=3$.
- **Laplace, Student-$t$, Cauchy, and Pareto innovations:** heavy-tailed alternatives; the middle AR hyperplane remains visible.
- **PIT view:** an empirical rank probability-integral transform of the process, together with its $k=2$ and interactive $k=3$ embeddings.

> An exact AR innovation hyperplane can be drawn only when the visible window contains all required lags, i.e. $p\leq k-1$. Thus $k=2$ supports an exact AR(1) line, while $k=3$ supports AR(1) and AR(2) planes. The notebook reports when a chosen order is too large for an exact overlay.

In [13]:
# If needed, uncomment the next line once:
# %pip install numpy scipy matplotlib plotly ipywidgets

import ast
import warnings

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, clear_output
from scipy.linalg import solve_discrete_lyapunov
from scipy.stats import chi2, rankdata

plt.style.use("seaborn-v0_8-whitegrid")

## Simulation and geometry functions

In [14]:
def companion_matrix(phi):
    """Companion matrix for an AR(p) coefficient vector."""
    phi = np.asarray(phi, dtype=float)
    p = len(phi)
    A = np.zeros((p, p))
    A[0, :] = phi
    if p > 1:
        A[1:, :-1] = np.eye(p - 1)
    return A


def is_stationary(phi):
    """Return stationarity flag and companion-matrix eigenvalues."""
    eigvals = np.linalg.eigvals(companion_matrix(phi))
    return bool(np.all(np.abs(eigvals) < 1)), eigvals


def parse_phi(text):
    """Parse entries such as '0.7', '0.6, -0.2', or '[0.6, -0.2]' safely."""
    text = text.strip()
    if not text:
        raise ValueError("Please enter at least one AR coefficient.")
    if text.startswith("[") or text.startswith("("):
        values = ast.literal_eval(text)
    else:
        values = [float(part.strip()) for part in text.split(",")]
    phi = np.atleast_1d(np.asarray(values, dtype=float))
    if phi.ndim != 1 or len(phi) == 0 or not np.all(np.isfinite(phi)):
        raise ValueError("AR coefficients must be a finite one-dimensional sequence.")
    return phi


def generate_ar_process(phi, n=1000, intercept=0.0, innovation="Gaussian",
                        innovation_sd=1.0, innovation_df=5.0, pareto_shape=2.5,
                        burnin=500, seed=7):
    """Simulate an AR(p); innovation_sd is an SD when finite, otherwise a scale."""
    phi = np.asarray(phi, dtype=float)
    stationary, eigvals = is_stationary(phi)
    if not stationary:
        raise ValueError(
            "The selected AR coefficients are not stationary. "
            f"Companion eigenvalue magnitudes: {np.round(np.abs(eigvals), 4)}"
        )
    if innovation_sd <= 0:
        raise ValueError("Innovation SD must be positive.")

    rng = np.random.default_rng(seed)
    p = len(phi)
    total = n + burnin + p
    if innovation.lower() == "gaussian":
        eps = rng.normal(0.0, innovation_sd, total)
    elif innovation.lower() == "uniform":
        half_width = np.sqrt(3.0) * innovation_sd
        eps = rng.uniform(-half_width, half_width, total)
    elif innovation.lower() == "laplace":
        eps = rng.laplace(0.0, innovation_sd / np.sqrt(2.0), total)
    elif innovation.lower() in {"student-t", "student t", "student"}:
        if innovation_df <= 2:
            raise ValueError("Student-t degrees of freedom must exceed 2 for a finite SD.")
        scale = innovation_sd * np.sqrt((innovation_df - 2.0) / innovation_df)
        eps = scale * rng.standard_t(innovation_df, total)
    elif innovation.lower() == "cauchy":
        eps = innovation_sd * rng.standard_cauchy(total)
    elif innovation.lower() == "pareto":
        if pareto_shape <= 1:
            raise ValueError("Pareto shape must exceed 1 so the innovations can be mean-centered.")
        raw = innovation_sd * (1.0 + rng.pareto(pareto_shape, total))
        eps = raw - innovation_sd * pareto_shape / (pareto_shape - 1.0)
    else:
        raise ValueError(
            "innovation must be Gaussian, Uniform, Laplace, Student-t, Cauchy, or Pareto."
        )

    x = np.zeros(total)
    stationary_mean = intercept / (1.0 - np.sum(phi))
    x[:p] = stationary_mean
    for t in range(p, total):
        x[t] = intercept + phi @ x[t - np.arange(1, p + 1)] + eps[t]
    return x[burnin + p:]


def forward_sliding_window(x, k):
    """Rows are (x_t, ..., x_{t+k-1})."""
    x = np.asarray(x)
    if not 1 <= k <= len(x):
        raise ValueError("k must lie between 1 and len(x).")
    return np.column_stack([x[j:len(x) - k + 1 + j] for j in range(k)])


def empirical_pit(x):
    """Rank-based empirical PIT in the open interval (0, 1)."""
    x = np.asarray(x)
    return (rankdata(x, method="average") - 0.5) / len(x)


def stationary_window_moments(phi, intercept, innovation_sd, k):
    """Theoretical mean and covariance of a Gaussian AR(p) k-window."""
    phi = np.asarray(phi, dtype=float)
    A = companion_matrix(phi)
    Q = np.zeros_like(A)
    Q[0, 0] = innovation_sd ** 2
    state_cov = solve_discrete_lyapunov(A, Q)
    e0 = np.zeros(len(phi))
    e0[0] = 1.0
    gamma = np.array([e0 @ np.linalg.matrix_power(A, h) @ state_cov @ e0
                      for h in range(k)])
    cov = np.fromfunction(lambda i, j: gamma[np.abs(i - j).astype(int)], (k, k))
    mean = np.full(k, intercept / (1.0 - np.sum(phi)))
    return mean, cov


def ar_surface(phi, intercept, x0, x1=None):
    """Last window coordinate under zero innovation."""
    if x1 is None:  # k=2, necessarily AR(1)
        return intercept + phi[0] * x0
    z = intercept + phi[0] * x1
    if len(phi) >= 2:
        z = z + phi[1] * x0
    return z


def ellipse_points(mean, cov, probability=0.95, n_points=300):
    values, vectors = np.linalg.eigh(cov)
    values = np.clip(values, 0, None)
    radius = np.sqrt(chi2.ppf(probability, df=2))
    angles = np.linspace(0, 2 * np.pi, n_points)
    circle = np.vstack((np.cos(angles), np.sin(angles)))
    return mean[:, None] + vectors @ np.diag(np.sqrt(values)) @ (radius * circle)


def ellipsoid_mesh(mean, cov, probability=0.95, n_u=50, n_v=28):
    values, vectors = np.linalg.eigh(cov)
    values = np.clip(values, 0, None)
    radius = np.sqrt(chi2.ppf(probability, df=3))
    u = np.linspace(0, 2 * np.pi, n_u)
    v = np.linspace(0, np.pi, n_v)
    sphere = np.stack([
        np.outer(np.cos(u), np.sin(v)),
        np.outer(np.sin(u), np.sin(v)),
        np.outer(np.ones_like(u), np.cos(v)),
    ], axis=0).reshape(3, -1)
    transformed = mean[:, None] + vectors @ np.diag(np.sqrt(values)) @ (radius * sphere)
    shape = (n_u, n_v)
    return tuple(transformed[i].reshape(shape) for i in range(3))

In [15]:
def plot_time_series(x, phi, innovation):
    fig, ax = plt.subplots(figsize=(12, 3.6))
    ax.plot(x, lw=1.0, color="#315a9a")
    ax.axhline(np.mean(x), color="black", ls="--", lw=1, alpha=0.7, label="sample mean")
    ax.set(title=f"AR({len(phi)}) process — {innovation} innovations",
           xlabel="time $t$", ylabel="$X_t$")
    robust_limits = np.quantile(x, [0.005, 0.995])
    if np.all(np.isfinite(robust_limits)) and robust_limits[0] < robust_limits[1]:
        margin = 0.08 * (robust_limits[1] - robust_limits[0])
        ax.set_ylim(robust_limits[0] - margin, robust_limits[1] + margin)
    ax.legend()
    plt.show()


def plot_embedding_k2(cloud, phi, intercept, innovation, innovation_sd, hdr_probability):
    fig, ax = plt.subplots(figsize=(7.2, 7.0))
    time_color = np.arange(len(cloud))
    points = ax.scatter(cloud[:, 0], cloud[:, 1], c=time_color, cmap="viridis",
                        s=10, alpha=0.55, label=r"$Y_t^{(2)}$")
    fig.colorbar(points, ax=ax, label="window time index $t$")

    lo, hi = np.quantile(cloud[:, 0], [0.005, 0.995])
    grid = np.linspace(lo, hi, 300)
    if len(phi) <= 1:
        middle = ar_surface(phi, intercept, grid)
        ax.plot(grid, middle, color="black", lw=2.2, label="middle AR hyperplane")
        if innovation.lower() == "uniform":
            bound = np.sqrt(3.0) * innovation_sd
            ax.plot(grid, middle + bound, color="#d1495b", lw=2, ls="--", label="border hyperplanes")
            ax.plot(grid, middle - bound, color="#d1495b", lw=2, ls="--")
    else:
        ax.text(0.02, 0.98, "Exact AR hyperplane unavailable: k=2 requires p≤1",
                transform=ax.transAxes, va="top", color="#a33", fontsize=10,
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="#a33"))

    if innovation.lower() == "gaussian":
        mean, cov = stationary_window_moments(phi, intercept, innovation_sd, k=2)
        ellipse = ellipse_points(mean, cov, hdr_probability)
        ax.plot(ellipse[0], ellipse[1], color="#d1495b", lw=2.5,
                label=f"{hdr_probability:.0%} Gaussian HDR ellipse")

    ax.set(title=r"Forward sliding-window points $Y_t^{(2)}$",
           xlabel="$X_t$", ylabel="$X_{t+1}$")
    x_limits = np.quantile(cloud[:, 0], [0.005, 0.995])
    y_limits = np.quantile(cloud[:, 1], [0.005, 0.995])
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(loc="best")
    plt.show()


def plot_embedding_k3(cloud, phi, intercept, innovation, innovation_sd, hdr_probability):
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=cloud[:, 0], y=cloud[:, 1], z=cloud[:, 2], mode="markers", name="Y<sub>t</sub><sup>(3)</sup>",
        marker=dict(
            size=2.5, color=np.arange(len(cloud)), colorscale="Viridis", opacity=0.55,
            colorbar=dict(title="time index t")
        )
    ))

    if len(phi) <= 2:
        q0 = np.quantile(cloud[:, 0], [0.01, 0.99])
        q1 = np.quantile(cloud[:, 1], [0.01, 0.99])
        g0, g1 = np.meshgrid(np.linspace(*q0, 28), np.linspace(*q1, 28))
        middle = ar_surface(phi, intercept, g0, g1)
        fig.add_trace(go.Surface(
            x=g0, y=g1, z=middle, name="middle AR hyperplane", showscale=False,
            opacity=0.34, colorscale=[[0, "#222222"], [1, "#222222"]],
            hovertemplate="middle AR hyperplane<extra></extra>"
        ))
        if innovation.lower() == "uniform":
            bound = np.sqrt(3.0) * innovation_sd
            for offset, label in [(bound, "upper border"), (-bound, "lower border")]:
                fig.add_trace(go.Surface(
                    x=g0, y=g1, z=middle + offset, name=label, showscale=False,
                    opacity=0.24, colorscale=[[0, "#d1495b"], [1, "#d1495b"]],
                    hovertemplate=label + "<extra></extra>"
                ))
    else:
        fig.add_annotation(
            text="Exact AR hyperplane unavailable: k=3 requires p≤2",
            x=0.01, y=0.99, xref="paper", yref="paper", showarrow=False,
            font=dict(color="#a33"), bgcolor="white"
        )

    if innovation.lower() == "gaussian":
        mean, cov = stationary_window_moments(phi, intercept, innovation_sd, k=3)
        ex, ey, ez = ellipsoid_mesh(mean, cov, hdr_probability)
        fig.add_trace(go.Surface(
            x=ex, y=ey, z=ez, name=f"{hdr_probability:.0%} Gaussian HDR ellipsoid",
            showscale=False, opacity=0.20,
            colorscale=[[0, "#d1495b"], [1, "#d1495b"]],
            hovertemplate=f"{hdr_probability:.0%} Gaussian HDR ellipsoid<extra></extra>"
        ))

    fig.update_layout(
        title="Forward sliding-window points Y<sub>t</sub><sup>(3)</sup> — drag to rotate",
        width=900, height=700,
        scene=dict(
            xaxis_title="$X_t$", yaxis_title="$X_{t+1}$", zaxis_title="$X_{t+2}$",
            xaxis=dict(range=np.quantile(cloud[:, 0], [0.005, 0.995]).tolist()),
            yaxis=dict(range=np.quantile(cloud[:, 1], [0.005, 0.995]).tolist()),
            zaxis=dict(range=np.quantile(cloud[:, 2], [0.005, 0.995]).tolist()),
            aspectmode="data"
        ),
        legend=dict(x=0.01, y=0.99),
        margin=dict(l=0, r=0, b=0, t=50)
    )
    fig.show()


def plot_pit_views(x):
    """Plot the empirical PIT process and its 2D and interactive 3D embeddings."""
    u = empirical_pit(x)
    pit2 = forward_sliding_window(u, 2)
    pit3 = forward_sliding_window(u, 3)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    axes[0].plot(u, lw=0.9, color="#6a4c93")
    axes[0].axhline(0.5, color="black", ls="--", lw=1, alpha=0.7)
    axes[0].set(title="Empirical PIT-transformed process", xlabel="time $t$", ylabel="$U_t$")
    axes[0].set_ylim(0, 1)

    pit_time_color = np.arange(len(pit2))
    pit_points = axes[1].scatter(pit2[:, 0], pit2[:, 1], c=pit_time_color,
                                 cmap="viridis", s=10, alpha=0.55)
    fig.colorbar(pit_points, ax=axes[1], label="window time index $t$")
    axes[1].axhline(0.5, color="black", ls="--", lw=1, alpha=0.5)
    axes[1].axvline(0.5, color="black", ls="--", lw=1, alpha=0.5)
    axes[1].set(title=r"PIT sliding-window points $Y_{U,t}^{(2)}$",
                xlabel="$U_t$", ylabel="$U_{t+1}$", xlim=(0, 1), ylim=(0, 1))
    axes[1].set_aspect("equal")
    plt.tight_layout()
    plt.show()

    fig3 = go.Figure(go.Scatter3d(
        x=pit3[:, 0], y=pit3[:, 1], z=pit3[:, 2], mode="markers",
        name="Y<sub>U,t</sub><sup>(3)</sup>",
        marker=dict(
            size=2.5, color=np.arange(len(pit3)), colorscale="Viridis", opacity=0.55,
            colorbar=dict(title="time index t")
        )
    ))
    fig3.update_layout(
        title="PIT sliding-window points Y<sub>U,t</sub><sup>(3)</sup> — drag to rotate",
        width=900, height=700,
        scene=dict(
            xaxis=dict(title="$U_t$", range=[0, 1]),
            yaxis=dict(title="$U_{t+1}$", range=[0, 1]),
            zaxis=dict(title="$U_{t+2}$", range=[0, 1]),
            aspectmode="cube"
        ),
        margin=dict(l=0, r=0, b=0, t=50)
    )
    fig3.show()
    return u, pit2, pit3


def run_experiment(phi, n, intercept, innovation, innovation_sd, innovation_df, pareto_shape, burnin,
                   seed, hdr_probability, max_points=2500):
    x = generate_ar_process(
        phi=phi, n=n, intercept=intercept, innovation=innovation,
        innovation_sd=innovation_sd, innovation_df=innovation_df,
        pareto_shape=pareto_shape, burnin=burnin, seed=seed
    )
    cloud2 = forward_sliding_window(x, 2)
    cloud3 = forward_sliding_window(x, 3)
    if len(cloud2) > max_points:
        idx2 = np.linspace(0, len(cloud2) - 1, max_points, dtype=int)
        cloud2 = cloud2[idx2]
    if len(cloud3) > max_points:
        idx3 = np.linspace(0, len(cloud3) - 1, max_points, dtype=int)
        cloud3 = cloud3[idx3]

    stationary, eigvals = is_stationary(phi)
    print(f"AR order p = {len(phi)}")
    print(f"Companion eigenvalues = {np.round(eigvals, 4)}")
    print(f"Sample mean = {x.mean():.4f}; sample SD = {x.std(ddof=1):.4f}")
    if innovation.lower() == "uniform":
        bound = np.sqrt(3.0) * innovation_sd
        print(f"Uniform innovation support = [{-bound:.4f}, {bound:.4f}]")
    elif innovation.lower() == "student-t":
        print(f"Student-t degrees of freedom = {innovation_df:.2f}")
    elif innovation.lower() == "cauchy":
        print(f"Cauchy scale = {innovation_sd:.4f}; mean and variance do not exist")
    elif innovation.lower() == "pareto":
        variance_text = "finite" if pareto_shape > 2 else "infinite"
        print(f"Centered Pareto shape = {pareto_shape:.2f}; variance is {variance_text}")

    plot_time_series(x, phi, innovation)
    plot_embedding_k2(cloud2, phi, intercept, innovation, innovation_sd, hdr_probability)
    plot_embedding_k3(cloud3, phi, intercept, innovation, innovation_sd, hdr_probability)
    u, pit2, pit3 = plot_pit_views(x)
    return x, cloud2, cloud3, u, pit2, pit3

## Interactive controls

Enter coefficients in lag order, for example `0.7` for AR(1) or `0.6, -0.2` for AR(2). For Gaussian, uniform, Laplace, and Student-$t$ innovations, the numeric scale control is the standard deviation $\sigma$. For Cauchy it is the Cauchy scale, and for Pareto it is the minimum-tail scale $x_m$. Pareto draws are shifted by their theoretical mean, so the shape parameter must exceed 1. The empirical PIT uses ranks and is therefore available even when the process marginal distribution has no closed form or finite moments.

In [ ]:
phi_widget = widgets.Text(value="0.7", description="AR coefficients:",
                          style={"description_width": "130px"}, layout=widgets.Layout(width="360px"))
innovation_widget = widgets.Dropdown(
    options=["Gaussian", "Uniform", "Laplace", "Student-t", "Cauchy", "Pareto"], value="Gaussian",
                                     description="Innovations:", style={"description_width": "130px"})
n_widget = widgets.IntSlider(value=1200, min=200, max=5000, step=100,
                             description="Sample size:", style={"description_width": "130px"})
sd_widget = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                                description="Scale / SD:", style={"description_width": "130px"})
df_widget = widgets.FloatSlider(value=5.0, min=2.1, max=30.0, step=0.1,
                                description="Student-t df:", style={"description_width": "130px"})
pareto_widget = widgets.FloatSlider(value=2.5, min=1.1, max=10.0, step=0.1,
                                    description="Pareto shape:", style={"description_width": "130px"})
intercept_widget = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1,
                                       description="Intercept:", style={"description_width": "130px"})
hdr_widget = widgets.FloatSlider(value=0.95, min=0.50, max=0.99, step=0.01,
                                 readout_format=".2f", description="HDR probability:",
                                 style={"description_width": "130px"})
seed_widget = widgets.IntText(value=7, description="Random seed:",
                              style={"description_width": "130px"})
run_button = widgets.Button(description="Generate and plot", button_style="primary", icon="play")
output = widgets.Output()


def handle_run(_=None):
    with output:
        clear_output(wait=True)
        try:
            phi = parse_phi(phi_widget.value)
            global x, cloud_k2, cloud_k3, u_pit, pit_cloud_k2, pit_cloud_k3
            x, cloud_k2, cloud_k3, u_pit, pit_cloud_k2, pit_cloud_k3 = run_experiment(
                phi=phi,
                n=n_widget.value,
                intercept=intercept_widget.value,
                innovation=innovation_widget.value,
                innovation_sd=sd_widget.value,
                innovation_df=df_widget.value,
                pareto_shape=pareto_widget.value,
                burnin=max(500, 10 * len(phi)),
                seed=seed_widget.value,
                hdr_probability=hdr_widget.value,
            )
        except Exception as exc:
            print(f"Could not generate the requested process: {exc}")


run_button.on_click(handle_run)
controls = widgets.VBox([
    widgets.HTML("<b>AR process settings</b>"),
    phi_widget,
    widgets.HBox([innovation_widget, sd_widget]),
    widgets.HBox([df_widget, pareto_widget]),
    widgets.HBox([n_widget, intercept_widget]),
    widgets.HBox([hdr_widget, seed_widget]),
    run_button,
])
display(controls, output)
handle_run()

Output()

## Non-widget alternative

If widgets are unavailable in your Jupyter frontend, edit and run the cell below directly.

In [ ]:
# Example: an AR(2) with uniform innovations.
# Uncomment to run.
# phi = np.array([0.6, -0.2])
# x, cloud_k2, cloud_k3, u_pit, pit_cloud_k2, pit_cloud_k3 = run_experiment(
#     phi=phi,
#     n=1500,
#     intercept=0.0,
#     innovation="Pareto",  # Gaussian, Uniform, Laplace, Student-t, Cauchy, or Pareto
#     innovation_sd=1.0,
#     innovation_df=4.0,        # used only for Student-t
#     pareto_shape=2.5,         # used only for Pareto; must exceed 1
#     burnin=500,
#     seed=7,
#     hdr_probability=0.95,
# )